# 🔬 Test Whisper Large V3 — Speech-to-Text
**Model:** `openai/whisper-large-v3`  
**Task:** Automatic Speech Recognition (ASR)  
**Doc:** [HF Whisper](https://huggingface.co/docs/transformers/main/en/model_doc/whisper)
---

## 1. Setup

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, pipeline
import time, os, psutil, io

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print(f'Device: {device} | Dtype: {dtype}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/(1024**3):.1f} GB')

## 2. Încărcare Model

In [ ]:
MODEL_ID = 'openai/whisper-large-v3'
mem_before = psutil.Process().memory_info().rss / (1024**2)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_ID, torch_dtype=dtype, low_cpu_mem_usage=True)
model = model.to(device)
model.eval()

mem_after = psutil.Process().memory_info().rss / (1024**2)

total_params = sum(p.numel() for p in model.parameters())
param_mb = sum(p.numel()*p.element_size() for p in model.parameters()) / (1024**2)

print(f'\n✅ Model încărcat!')
print(f'   Parametri: {total_params:,}')
print(f'   Dimensiune: {param_mb:.0f} MB')
print(f'   RAM delta: {mem_after-mem_before:.0f} MB')
print(f'   Encoder layers: {model.config.encoder_layers}')
print(f'   Decoder layers: {model.config.decoder_layers}')
print(f'   d_model: {model.config.d_model}')
print(f'   Vocab size: {model.config.vocab_size}')

## 3. Mock Audio Data
Generăm semnale audio sintetice. **🔄 Înlocuiește cu fișiere audio reale.**

In [ ]:
SAMPLE_RATE = 16000  # Whisper necesită 16kHz

def make_sine(freq, duration, sr=SAMPLE_RATE):
    t = np.linspace(0, duration, int(sr*duration), dtype=np.float32)
    return np.sin(2*np.pi*freq*t)

def make_speech_like(duration, sr=SAMPLE_RATE):
    """Simulare speech: mix de frecvențe 100-3000Hz + variație amplitudine."""
    t = np.linspace(0, duration, int(sr*duration), dtype=np.float32)
    signal = np.zeros_like(t)
    for f in [150, 300, 600, 1200, 2400]:
        signal += np.sin(2*np.pi*f*t) * np.random.uniform(0.1, 0.5)
    env = np.abs(np.sin(2*np.pi*2*t))  # envelope
    return (signal * env / np.max(np.abs(signal))).astype(np.float32)

def make_silence(duration, sr=SAMPLE_RATE):
    return np.zeros(int(sr*duration), dtype=np.float32)

def make_noise(duration, sr=SAMPLE_RATE):
    np.random.seed(42)
    return (np.random.randn(int(sr*duration)) * 0.3).astype(np.float32)

# ============================================================
# 🔄 MOCK DATA — Înlocuiește:
#    import soundfile as sf
#    audio, sr = sf.read('fisier.wav')
#    mock_audio['Nume'] = {'audio': audio, 'sr': sr, 'expected': 'textul așteptat'}
# ============================================================
mock_audio = {
    'Ton 440Hz (2s)': {'audio': make_sine(440, 2.0), 'sr': SAMPLE_RATE, 'expected': ''},
    'Speech-like (3s)': {'audio': make_speech_like(3.0), 'sr': SAMPLE_RATE, 'expected': ''},
    'Silence (2s)': {'audio': make_silence(2.0), 'sr': SAMPLE_RATE, 'expected': ''},
    'Noise (2s)': {'audio': make_noise(2.0), 'sr': SAMPLE_RATE, 'expected': ''},
}

# Vizualizare
fig, axes = plt.subplots(len(mock_audio), 2, figsize=(14, 3*len(mock_audio)))
for idx, (name, d) in enumerate(mock_audio.items()):
    t = np.arange(len(d['audio'])) / d['sr']
    axes[idx,0].plot(t, d['audio'], linewidth=0.5, color='#2196F3')
    axes[idx,0].set_title(f'{name} — Waveform', fontsize=9); axes[idx,0].set_ylabel('Amp')
    axes[idx,1].specgram(d['audio'], Fs=d['sr'], NFFT=512, noverlap=256, cmap='magma')
    axes[idx,1].set_title(f'{name} — Spectrogram', fontsize=9); axes[idx,1].set_ylabel('Hz')
plt.tight_layout(); plt.show()

## 4. Funcții Metrici ASR

In [ ]:
def word_error_rate(ref, hyp):
    """WER — Word Error Rate. Mai mic = mai bine."""
    if not ref: return 0.0 if not hyp else 1.0
    r, h = ref.split(), hyp.split()
    d = np.zeros((len(r)+1, len(h)+1), dtype=int)
    for i in range(len(r)+1): d[i][0] = i
    for j in range(len(h)+1): d[0][j] = j
    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+(0 if r[i-1]==h[j-1] else 1))
    return d[len(r)][len(h)] / len(r)

def char_error_rate(ref, hyp):
    """CER — Character Error Rate. Mai mic = mai bine."""
    if not ref: return 0.0 if not hyp else 1.0
    r, h = list(ref), list(hyp)
    d = np.zeros((len(r)+1, len(h)+1), dtype=int)
    for i in range(len(r)+1): d[i][0] = i
    for j in range(len(h)+1): d[0][j] = j
    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+(0 if r[i-1]==h[j-1] else 1))
    return d[len(r)][len(h)] / len(r)

def audio_stats(audio):
    return {'duration_s': len(audio)/SAMPLE_RATE, 'rms': float(np.sqrt(np.mean(audio**2))),
            'peak': float(np.max(np.abs(audio))), 'mean': float(np.mean(audio)),
            'std': float(np.std(audio)), 'silence_%': float(np.sum(np.abs(audio)<0.01)/len(audio)*100)}

print('✅ Metrici ASR definite: WER, CER, audio stats')

## 5. Transcriere + Metrici

In [ ]:
results = {}
for name, d in mock_audio.items():
    print(f'\n{"="*60}')
    print(f'🔄 {name}')
    audio, sr = d['audio'], d['sr']
    
    # Pre-procesare
    inputs = processor(audio, sampling_rate=sr, return_tensors='pt')
    input_features = inputs.input_features.to(device, dtype=dtype)
    
    # Inferență
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    with torch.no_grad():
        generated_ids = model.generate(input_features, max_new_tokens=256)
    t_inf = time.perf_counter() - t0
    
    # Decodare
    transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    
    # Metrici
    expected = d['expected']
    wer = word_error_rate(expected, transcription) if expected else None
    cer = char_error_rate(expected, transcription) if expected else None
    a_stats = audio_stats(audio)
    gpu_peak = torch.cuda.max_memory_allocated()/(1024**2) if torch.cuda.is_available() else 0
    
    results[name] = {
        'transcription': transcription, 'expected': expected,
        'wer': wer, 'cer': cer,
        'time_s': t_inf, 'gpu_peak_mb': gpu_peak,
        'rtf': t_inf / a_stats['duration_s'],  # Real-Time Factor
        'audio_stats': a_stats,
        'num_tokens': len(generated_ids[0]),
    }
    
    print(f'   Durată audio: {a_stats["duration_s"]:.1f}s')
    print(f'   Transcriere: "{transcription}"')
    if expected: print(f'   Așteptat:    "{expected}"')
    if wer is not None: print(f'   WER: {wer:.4f} | CER: {cer:.4f}')
    print(f'   Timp: {t_inf:.3f}s | RTF: {results[name]["rtf"]:.3f}x | Tokens: {results[name]["num_tokens"]}')

print(f'\n{"="*60}\n✅ Toate procesate!')

## 6. Rezultate Transcriere

In [ ]:
print('='*90)
print('REZULTATE TRANSCRIERE')
print('='*90)
print(f'{"Clip":<25} {"Tokens":>7} {"Transcriere"}')
print(f'{"─"*90}')
for name, r in results.items():
    t = r['transcription'][:50] + ('...' if len(r['transcription'])>50 else '')
    print(f'{name:<25} {r["num_tokens"]:>7} "{t}"')

## 7. Metrici WER/CER (dacă ai expected text)

In [ ]:
has_expected = any(r['expected'] for r in results.values())
if has_expected:
    print('='*80)
    print(f'{"Clip":<25} {"WER":>8} {"CER":>8}')
    print(f'{"─"*45}')
    wers, cers = [], []
    for name, r in results.items():
        if r['wer'] is not None:
            print(f'{name:<25} {r["wer"]:>8.4f} {r["cer"]:>8.4f}')
            wers.append(r['wer']); cers.append(r['cer'])
    if wers:
        print(f'{"─"*45}')
        print(f'{"MEDIE":<25} {np.mean(wers):>8.4f} {np.mean(cers):>8.4f}')
else:
    print('⚠️ Nu sunt texte expected definite — WER/CER nu se pot calcula.')
    print('   Adaugă expected text în mock_audio din secțiunea 3 pt metrici cu referință.')

## 8. Performanță

In [ ]:
print('='*95)
print('PERFORMANȚĂ')
print('='*95)
print(f'{"Clip":<25} {"Audio (s)":>10} {"Inf (s)":>10} {"RTF":>8} {"Tokens":>8} {"GPU Peak":>10}')
print(f'{"─"*75}')
times = []
for name, r in results.items():
    a = r['audio_stats']
    times.append(r['time_s'])
    print(f'{name:<25} {a["duration_s"]:>10.1f} {r["time_s"]:>10.3f} {r["rtf"]:>8.3f} {r["num_tokens"]:>8} {r["gpu_peak_mb"]:>8.0f} MB')
print(f'{"─"*75}')
print(f'{"MEDIE":<25} {"":>10} {np.mean(times):>10.3f}')
print(f'\nRTF < 1.0 = mai rapid decât real-time')

## 9. Statistici Audio

In [ ]:
print('='*90)
print('STATISTICI AUDIO')
print('='*90)
print(f'{"Clip":<25} {"Durată":>8} {"RMS":>8} {"Peak":>8} {"StdDev":>8} {"Silence%":>10}')
print(f'{"─"*70}')
for name, r in results.items():
    a = r['audio_stats']
    print(f'{name:<25} {a["duration_s"]:>7.1f}s {a["rms"]:>8.4f} {a["peak"]:>8.4f} {a["std"]:>8.4f} {a["silence_%"]:>9.1f}%')

## 10. Grafice

In [ ]:
names = list(results.keys())
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Timp inferență
axes[0].barh(names, [results[n]['time_s'] for n in names], color='#4CAF50')
axes[0].set_xlabel('Secunde'); axes[0].set_title('Timp Inferență', fontweight='bold')

# RTF
rtfs = [results[n]['rtf'] for n in names]
colors = ['#4CAF50' if r<1 else '#FF9800' for r in rtfs]
axes[1].barh(names, rtfs, color=colors)
axes[1].axvline(x=1.0, color='red', linestyle='--', alpha=0.7, label='Real-time')
axes[1].set_xlabel('RTF'); axes[1].set_title('Real-Time Factor (< 1 = rapid)', fontweight='bold')
axes[1].legend()

# Tokens generate
axes[2].barh(names, [results[n]['num_tokens'] for n in names], color='#2196F3')
axes[2].set_xlabel('Tokens'); axes[2].set_title('Tokens Generate', fontweight='bold')

plt.suptitle('Whisper Large V3 — Performanță', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 11. Sumar

In [ ]:
print('█'*70)
print('  SUMAR — Whisper Large V3')
print('█'*70)
print(f'  Model:     {MODEL_ID}')
print(f'  Parametri: {total_params:,}')
print(f'  Size:      {param_mb:.0f} MB')
print(f'  Device:    {device}')
print(f'  Timp mediu inferență: {np.mean(times):.3f}s')
print(f'  RTF mediu: {np.mean([results[n]["rtf"] for n in results]):.3f}x')
if has_expected and wers:
    print(f'  WER mediu: {np.mean(wers):.4f}')
    print(f'  CER mediu: {np.mean(cers):.4f}')
print(f'\n  Notă: Mock audio = semnale sintetice, nu speech real.')
print(f'  Înlocuiește cu fișiere audio reale pt metrici relevante.')
print('█'*70)

## 12. Salvare

In [ ]:
OUT = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'output_whisper')
os.makedirs(OUT, exist_ok=True)
with open(os.path.join(OUT, 'transcriptions.txt'), 'w', encoding='utf-8') as f:
    for name, r in results.items():
        f.write(f'{name}:\n  Transcription: {r["transcription"]}\n  Time: {r["time_s"]:.3f}s\n  RTF: {r["rtf"]:.3f}\n\n')
print(f'✅ Salvat în: {OUT}')

---
## 📝 Note
- **Mock data:** Înlocuiește `mock_audio` cu `soundfile.read('fisier.wav')` + `expected` text
- **WER/CER:** Se calculează doar dacă ai `expected` text (ground truth)
- **RTF:** Real-Time Factor — < 1.0 înseamnă mai rapid decât real-time
- **Limbi:** Whisper e multilingv — setează `language='ro'` în `model.generate()` pt română
- **VRAM:** ~6-10 GB pt large-v3. Folosește `torch.float16` pt mai puțin